# Develop and test a quantum error correction scheme with `qdk.ec`

Taking a quantum error correction scheme from a paper to a declarative artifact is hard. Checks, readouts, and circuit semantics must stay consistent as the design changes.

`qodec` owns the declarative artifact and its persistence. `qdk.ec` adds three focused workflows:

| workflow | API | question |
| --- | --- | --- |
| derive | `ec.derive` | Which checks and readout bindings follow from exact simulation? |
| profile | `ec.GadgetProfile`, `ec.SubsystemCode` | What does this gadget or code do? |
| audit | `ec.audit` | Is the complete protocol internally consistent? |

## Installing

`qdk.ec` is an optional extra of the `qdk` package:

```bash
pip install "qdk[ec]"
```

## 1. Load a qodec

The `qodec` package moves qodecs between disk and memory. Start from `c4.qodec.yaml`, next to this notebook. It describes the [[4,2,2]] error-detecting code, which encodes two logical qubits in four physical qubits and detects any single-qubit fault.

In [ ]:
import qodec as qc
import qdk.ec as ec

protocol = qc.Qodec.load("c4.qodec.yaml")
print(protocol.summary())

A qodec is a chain of **layers**, from the most abstract instruction set down to
the most concrete. Each layer carries the **gadgets** that lower one of its
instructions into a circuit over the layer below. Here there is a single lowering
edge: the logical `C4` instruction set down to physical `stim` operations.

In [ ]:
layer = protocol.layers[0]
print("lowering:", layer.isa.name, "->", protocol.layers[1].isa.name)
print("gadgets: ", sorted(layer.gadgets))

## 2. Profile the code and gadgets

`SubsystemCode` adds algebraic analysis to qodec's code data. `GadgetProfile` reports facts obtained through exact simulation.

In [ ]:
code = ec.SubsystemCode.of(protocol.codes["C4"])

print("stabilizers:", list(code.stabilizers))
print("logical basis:", list(code.logical_basis))

distance, witness = code.distance()
print(f"distance: {distance} (witness: {[str(p) for p in witness]})")

Distance 2 is exactly what "error *detecting*" means: there is a weight-2 logical
error, so a single fault is always visible but never correctable.

### Declared vs. realized action

Every gadget makes a promise — the action of the instruction it `implements` — and
keeps it with a circuit. Those are two independent objects, and `qdk.ec` can
compute both and compare them. This is the check that catches a transcription slip
between the paper and the circuit.

In [ ]:
measure_zz = layer.gadgets["measure_zz"]
profile = ec.GadgetProfile(measure_zz)

print("objective:", profile.objective)
print("action:   ", profile.action)
print("mismatch: ", profile.action.why_not_equivalent_to(profile.objective) or "none")

### Checks and readouts

A gadget's circuit produces raw measurement outcomes. Two derived structures give
those outcomes meaning:

* **checks** — parities of outcomes that are *deterministic*, so a flip signals a
  fault. These are what a decoder consumes.
* **readouts** — the parities that carry the logical answer the instruction
  promised.

Both are discovered by exact simulation, so you never have to derive them by
hand.

In [ ]:
print("checks:  ", profile.checks)
print("readouts:", profile.readouts)

## 3. Derive checks and readouts

Checks and readouts are derivable, so an author does not need to write them. `ec.derive` accepts either one gadget or a complete qodec and returns a new artifact, leaving the input unchanged.

In [ ]:
draft = qc.Gadget(
    measure_zz.implements,
    measure_zz.circuit,
    inputs=list(measure_zz.inputs),
    outputs=list(measure_zz.outputs),
    checks=[],
    readouts=list(measure_zz.readouts),
)
print("draft checks:    ", list(draft.checks))

completed = ec.derive(draft)
print("completed checks:", list(completed.checks))

The same function derives every gadget in a qodec. It returns a new protocol and never mutates the input.

In [ ]:
completed_protocol = ec.derive(protocol)

for mnemonic, gadget in sorted(completed_protocol.layers[0].gadgets.items()):
    print(f"{mnemonic:16s} {len(gadget.checks)} check(s)")

### Save with qodec

Persistence stays on the artifact type. `Qodec.save` writes the protocol in qodec's native format, and `Qodec.load` reads it back.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    path = Path(directory) / "completed.qodec.yaml"
    completed_protocol.save(str(path), single_file=True)
    reloaded = qc.Qodec.load(str(path))

print("round-trips:", reloaded.name == completed_protocol.name)

## 4. Audit the qodec

`ec.audit` runs the complete rule set and returns every diagnostic. Each diagnostic identifies the rule, artifact, and reason. Filter the report through its properties when only one severity matters.

In [ ]:
report = ec.audit(protocol)
print(f"{len(report.errors)} error(s), {len(report.warnings)} warning(s)")

for diagnostic in report.errors + report.warnings[:2]:
    print()
    print(f"[{diagnostic.severity.name}] {diagnostic.rule}")
    print(f"  {diagnostic.summary}")

The report catches mismatched readouts and incomplete output frames that are difficult to see in a paper but fatal in a compilation pipeline.

### Compare gadgets

Profiles own semantic comparison. This keeps the comparison next to the simulated action and provides an explanation when two gadgets differ.

In [ ]:
measure_xx = ec.GadgetProfile(layer.gadgets["measure_xx"])

print("measure_zz == itself:    ", profile.is_equivalent_to(profile))
print("measure_zz == measure_xx:", profile.is_equivalent_to(measure_xx))
print("why not:", profile.why_not_equivalent_to(measure_xx))

## Where to go next

* Use `qodec` to load and save protocols.
* Use `ec.derive` and `ec.build_qodec` to produce new artifacts.
* Use `ec.GadgetProfile` and `ec.SubsystemCode` for semantic analysis.
* Use `ec.audit` to validate a complete protocol.

The resulting qodec remains ordinary data that a downstream compilation pipeline can consume without another representation.